# Milestone 2: RAG Pipeline Exploration

This notebook explores a Retrieval-Augmented Generation (RAG) pipeline
for Amazon Digital Music reviews.

We evaluate:
- Retrieval quality
- Prompt design
- Generated responses

In [1]:
import sys
import os

# Go one level up from notebooks → project root
sys.path.append(os.path.abspath(".."))

In [2]:
from src.semantic import load_faiss
from src.rag_pipeline import run_rag_pipeline, load_llm
from src.prompts import build_rag_prompt

c:\Users\Shruti\miniforge3\envs\search-app\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
vector_store = load_faiss()
llm = load_llm()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3189.00it/s]


In [4]:
query = "What is a good relaxing album?"

result = run_rag_pipeline(query, vector_store=vector_store, llm=llm)

print("Answer:\n", result["answer"])

Answer:
 Based on the reviews, I recommend "Tapestries: The Music of Paul Simon" or "The Definitive America" for relaxing music. If you're looking for a collection of inspiring classics, consider "Beside Still Waters Presents Inspiring Classics".


In [5]:
print("Context:\n")
print(result["context"])

Context:

[Document 1]
Product ASIN: B000031W4K
Product Title: Tapestries: The Music of Paul Simon
Rating: 5.0
Review Title: Beautiful Music!
Review Text: great selection of relaxing songs.
Similarity Score: 0.4008

[Document 2]
Product ASIN: B000CIKTXS
Product Title: Beside Still Waters Presents Inspiring Classics
Rating: 5.0
Review Title: Five Stars
Review Text: more of the collection of relaxing songs
Similarity Score: 0.4185

[Document 3]
Product ASIN: B0BCXFBDYR
Product Title: The Definitive America
Rating: 4.0
Review Title: Good 70:s soft rock band.
Review Text: nice group of songs for relaxing.
Similarity Score: 0.4464


Prompt Variant 1 — Basic

In [6]:
def prompt_v1(query, context):
    return f"""
Answer the question based on the context.

Context:
{context}

Question:
{query}

Answer:
"""

Prompt Variant 2 — Strict

In [7]:
def prompt_v2(query, context):
    return f"""
You are a helpful assistant answering questions using ONLY the given context.

If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{query}

Answer clearly and concisely:
"""

Prompt Variant 3 — Structured

In [8]:
def prompt_v3(query, context):
    return f"""
You are a music recommendation assistant.

Use the context to answer the query. If exact details (like song names)
are not present, give a helpful recommendation based on the album or artist.

Rules:
- Prefer album or artist names if song names are missing
- You may infer general recommendations
- Keep answer under 200 characters
- Do NOT say "I don't know" unless context is completely irrelevant

Context:
{context}

Question:
{query}

Answer:
"""

In [9]:
query = "Which albums are good for studying?"

base = run_rag_pipeline(query, vector_store=vector_store, llm=llm)

context = base["context"]

for i, prompt_fn in enumerate([prompt_v1, prompt_v2, prompt_v3], start=1):
    print(f"\n--- Prompt Variant {i} ---\n")
    
    prompt = prompt_fn(query, context)
    
    from src.rag_pipeline import generate_answer
    answer = generate_answer(prompt, llm)
    
    print(answer)


--- Prompt Variant 1 ---

Based on the provided context, it does not mention any albums being specifically good for studying. However, the reviews for the albums are generally positive.

If we consider the genres of the albums (e.g., Sammy Davis Jr. Greatest Hits, Vol. 2 is a collection of popular songs, and Rockers For Lovers is a rock music compilation), they may not be the ideal choice for studying due to their entertainment-focused nature.

However, if we had to choose based on the provided information, we might consider that none of the mentioned albums are specifically designed for studying.

--- Prompt Variant 2 ---

I don't know.

--- Prompt Variant 3 ---

Based on the provided context, the albums that might be suitable for studying are those with a mellow or nostalgic vibe, which can help create a focus-friendly atmosphere.

I recommend checking out Lou Reed or T. Rex's albums, as they are often characterized by a more laid-back, old-school rock sound. Their music might provi

In [10]:
queries = [
    "What are relaxing albums?",
    "Best music for studying?",
    "Good workout music?",
]

for q in queries:
    print(f"\n=== Query: {q} ===\n")
    
    result = run_rag_pipeline(q, vector_store=vector_store, llm=llm)
    print(result["answer"])


=== Query: What are relaxing albums? ===

Based on the context, here are some relaxing album recommendations:

1. Ambient Dream Lounge (you've already mentioned it, but it's a great choice)
2. Tapestries: The Music of Paul Simon (a soothing collection of acoustic and folk songs)
3. Beside Still Waters Presents Inspiring Classics (a calming compilation of classics)
4. Brian Eno's Ambient albums (e.g., 'Ambient 1: Music for Airports')
5. Max Richter's 'Sleep' (a calm, 8-hour long album)

=== Query: Best music for studying? ===

Based on the reviews, it seems like the music is background and enjoyable. I recommend the album "The Nervous Man" as it has the highest similarity score. However, if you're looking for a specific genre or style, I'd suggest exploring the works of artists similar to those found on "Rockers For Lovers".

=== Query: Good workout music? ===

Based on the reviews, I recommend upbeat Christian music like "Tuesday's Child" or popular dance tracks from "Hits 2013 Vol. 2

Failure Case

In [11]:
query = "What is the best phone?"

result = run_rag_pipeline(query, vector_store=vector_store, llm=llm)

print(result["answer"])

Based on the context, it seems you're looking for a phone that can withstand drops, but the reviews are actually about phone protectors. However, since XTC is mentioned, I'll recommend an artist: The Beatles. They have a wide range of songs and albums that are perfect for various moods.


## Observations

### Prompt Comparison

**Prompt Variant 1 (Basic Prompt):**
- Generates answers even when context is weak or irrelevant.
- Tends to make assumptions (e.g., inferring study suitability from morning routines).
- Higher risk of **hallucination**.
- Useful for exploratory responses but less reliable.

**Prompt Variant 2 (Strict Prompt):**
- Most **conservative and reliable**.
- Correctly outputs *"I don't know"* when the answer is not supported by context.
- Minimizes hallucinations.
- Best suited for applications where **accuracy and grounding** are important.

**Prompt Variant 3 (Structured Prompt):**
- Produces more **interpretable and structured answers**.
- Attempts to extract meaningful patterns (e.g., mellow tone → suitable for studying).
- Slightly more informative than Variant 2 but may still infer beyond explicit context.
- Good balance between informativeness and structure.

---

### Retrieval Quality

- Retrieval performs well for **domain-relevant queries** such as:
  - "What are relaxing albums?"
  - Successfully extracts albums like *Wolf in Sheep's Clothing*, *First Snow*, and *Oceanside Piano*.

- Retrieval struggles when:
  - Query intent is **not explicitly represented** in reviews (e.g., "studying", "workout").
  - Leads to either:
    - weak inference (Prompt V1, V3), or
    - correct abstention (Prompt V2).

---

### Query Difficulty Observations

- **Easy Queries:**
  - Queries directly aligned with review language (e.g., "relaxing albums").
  - Clear signal in text → strong RAG performance.

- **Medium Difficulty:**
  - Queries requiring light interpretation (e.g., "studying music").
  - Performance depends heavily on prompt design.

- **Hard Queries:**
  - Queries with **no direct evidence** in context (e.g., "workout music").
  - System either fails gracefully (V2) or produces minimal/weak output.

---

### Overall Findings

- Prompt design significantly impacts RAG performance.
- **Prompt Variant 2** is the most reliable for factual grounding.
- **Prompt Variant 3** provides better readability and structured output.
- Retrieval is effective but limited by the **semantic coverage of the dataset**.

---

### Key Takeaways

- Strong prompt constraints reduce hallucination.
- RAG systems are highly dependent on:
  - quality of retrieved documents
  - alignment between query and dataset
- Combining **good retrieval + strict prompting** yields the best results.

## LLM Comparison: Llama 3.1 8B vs Gemma 2 9B

Testing both models on 5 identical queries using the same retrieved context and prompt (Variant 3).
The goal is to understand how model quality and architecture affect RAG output quality.

In [16]:
from groq import Groq
import os

# Initialize both models via Groq
client_llama = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Initialize both models via Groq
client_llama = Groq(api_key=os.getenv("GROQ_API_KEY"))
models = client_llama.models.list()
for m in sorted(models.data, key=lambda x: x.id):
    print(m.id)

allam-2-7b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
groq/compound
groq/compound-mini
llama-3.1-8b-instant
llama-3.3-70b-versatile
meta-llama/llama-4-scout-17b-16e-instruct
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
qwen/qwen3-32b
whisper-large-v3
whisper-large-v3-turbo


In [24]:
client_gemma = Groq(api_key=os.getenv("GROQ_API_KEY"))


LLAMA_INSTANT_MODEL = "llama-3.1-8b-instant"
LLAMA_VERSATILE_MODEL = "llama-3.3-70b-versatile"

def query_model(model_name, prompt):
    response = client_llama.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=200
    )
    return response.choices[0].message.content.strip()

In [25]:
from src.rag_pipeline import build_context, retrieve_documents
from src.prompts import build_rag_prompt

QUERIES = [
    "What is a good album for studying?",
    "Recommend a relaxing jazz album",
    "Best Taylor Swift album according to reviews",
    "Music for a long road trip",
    "Upbeat album for working out",
]

for query in QUERIES:
    # Same context and prompt for both models
    docs_with_scores = retrieve_documents(query, vector_store, k=5)
    context = build_context(docs_with_scores, max_docs=3)
    prompt  = build_rag_prompt(query, context)

    llama_instant_answer = query_model(LLAMA_INSTANT_MODEL, prompt)
    llama_versatile_answer = query_model(LLAMA_VERSATILE_MODEL, prompt)

    print(f"Query: {query} \n")
    print(f"  Llama Instant : {llama_instant_answer}")
    print(f"  Llama Versatile : {llama_versatile_answer}")
    print()

Query: What is a good album for studying? 

  Llama Instant : Based on the context, I'd recommend "Lyrical Lion" by Klondike Kat. It's a laid-back album that might help you relax and focus while studying.
  Llama Versatile : Tubular Bells LP by Mike Oldfield.

Query: Recommend a relaxing jazz album 

  Llama Instant : Based on your reviews, I recommend "The Earl Klugh Trio Volume One" or the "Best of so Far" album. Both are highly rated for relaxing and smooth jazz experiences.
  Llama Versatile : The Earl Klugh Trio Volume One or similar albums by Earl Klugh.

Query: Best Taylor Swift album according to reviews 

  Llama Instant : Based on the reviews, the best Taylor Swift album according to the reviewers is "Fearless Taylor's Version". It has the highest similarity score (0.6126) and all reviews for this album are 5-star ratings, indicating a highly positive opinion.
  Llama Versatile : Fearless Taylor's Version

Query: Music for a long road trip 

  Llama Instant : Based on your re